In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, BertForSequenceClassification
from datasets import load_dataset, Dataset

2025-02-28 11:29:34.855745: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-28 11:29:34.870512: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1740722374.889622  488347 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740722374.896049  488347 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-28 11:29:34.915988: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
df_train = pd.read_csv("Train_title.csv", encoding = 'utf-8')

In [3]:
df_train.head(10)

,news_category,news_title
0,Tamil Nadu,மாவோயிஸ்ட் தாக்குதலில் வீரமரணம் : துப்பாக்கி க...
1,Tamil Nadu,காஞ்சிபுரத்தில் இன்று அதிகாலை மளிகை கடை குடோனி...
2,Cinema,கிளிப்பிங்ஸ்
3,World,அமெரிக்காவின் டெக்சாஸ் மாகாணம்
4,Sports,விளையாட்டு அமைச்சகம் பரிந்துரை பி.வி.சிந்துவுக...
5,Crime,எழும்பூரில் உள்ள வணிக வளாகத்தில் 14 கடைகளை உடை...
6,Sports,டோனியுடன் உரசல் இல்லை
7,District News,தொன்னாடு ஊராட்சியில் வங்கி தலைவர் தேர்வு
8,India,நடிகை வீட்டில் கள்ள நோட்டுக்கள்
9,Tamil Nadu,ஜனநாயக உரிமையை காலில் போட்டு ஜெயலலிதா அரசு மித...


In [4]:
df_train.dtypes

news_category    object
news_title       object
dtype: object

In [5]:
df_test = pd.read_csv("Test_title.csv", encoding = 'utf-8')

In [6]:
df_test.head(10)

,news_category,news_title
0,Sports,இந்தியர்களுக்கு ‘கிரிக்கெட்டிஸ்’ நோய்: ராம்...
1,World,நைஜீரிய தீவிரவாதிகள் அட்டகாசம் துப்பாக்கி சூட்...
2,India,அரசியலில் இருந்து மோடியை விரட்டும் வரை ஓயமாட்...
3,Sports,‘டெஸ்ட் கிரிக்கெட்தான் சவாலானது’: ராகுல் டிராவ...
4,World,பாகிஸ்தானில் பரபரப்பு அதிபர் ஒபாமா படம் போட்ட ...
5,Cinema,டால்பினை தத்தெடுத்தார் எமி ஜாக்ஸன்
6,Cinema,பிரதமர் விருந்து: தனுஷ் மகிழ்ச்சி
7,World,தமிழர் வசிக்கும் பகுதியில் அடுத்த ஆண்டில் தேர்...
8,Sports,ஏஎப்சி கோப்பை பைனலில் பெங்களூரு மற்ற போட்டிகளை...
9,District News,மதுராந்தகம் அருகே இன்ஜினியரிங் கல்லூரி மாணவர் ...


In [7]:
df_test

,news_category,news_title
0,Sports,இந்தியர்களுக்கு ‘கிரிக்கெட்டிஸ்’ நோய்: ராம்...
1,World,நைஜீரிய தீவிரவாதிகள் அட்டகாசம் துப்பாக்கி சூட்...
2,India,அரசியலில் இருந்து மோடியை விரட்டும் வரை ஓயமாட்...
3,Sports,‘டெஸ்ட் கிரிக்கெட்தான் சவாலானது’: ராகுல் டிராவ...
4,World,பாகிஸ்தானில் பரபரப்பு அதிபர் ஒபாமா படம் போட்ட ...
...,...,...
12802,Sports,பாக்.கை சுருட்டியது : பைனலில் இலங்கை
12803,Crime,குடியிருப்புகளுக்கு மத்தியில் இயங்கிய போலி மது...
12804,District News,பாமக நிர்வாகியை கொல்ல முயற்சி போலீசில் தம்பி ப...
12805,Crime,ஜி.ஹெச்சில் ரவுடி கொலை : 8 பேர் கும்பல் கைது


In [8]:
df_test['news_title'][6]

'பிரதமர் விருந்து:  தனுஷ் மகிழ்ச்சி'

In [9]:
classes = set(df_train['news_category'].values)
display(classes)

{'Cinema', 'Crime', 'District News', 'India', 'Sports', 'Tamil Nadu', 'World'}

In [10]:
label_mapping = {category: code for code, category in enumerate(sorted(set(df_train['news_category'].values)))}
print(label_mapping)

{'Cinema': 0, 'Crime': 1, 'District News': 2, 'India': 3, 'Sports': 4, 'Tamil Nadu': 5, 'World': 6}


In [11]:
df_train['label'] = df_train['news_category'].map(label_mapping)
df_test['label'] = df_test['news_category'].map(label_mapping)

In [12]:
df_train.columns

Index(['news_category', 'news_title', 'label'], dtype='object')

In [13]:
df_train[['news_category', 'label']].head(5)

,news_category,label
0,Tamil Nadu,5
1,Tamil Nadu,5
2,Cinema,0
3,World,6
4,Sports,4


In [15]:
df_train = df_train[['news_title', 'label']]
df_test = df_test[['news_title', 'label']]

In [16]:
df_train.columns

Index(['news_title', 'label'], dtype='object')

In [17]:
classes_num = df_train['label'].nunique()
ds_t = Dataset.from_pandas(df_train)
ds_v = Dataset.from_pandas(df_test)

In [22]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
import os

# Model training parameters
max_sequence_length = 64
epochs = 5
batch_size = 64
save_steps = 1000

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# Training function
def train_model(model_name):
    log_file = f"{model_name}_results.csv"
    
    # Initialize logging
    with open(log_file, 'w') as f:
        f.write("Model,Accuracy,F1\n")
    
    print(f"\nTraining Model: {model_name}")
    
    # Set seed for reproducibility
    set_seed(42)

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=classes_num).to('cuda')
    
    # Preprocessing function
    def preprocess_function(examples):
        return tokenizer(examples['news_title'], truncation=True, padding="max_length", max_length=max_sequence_length)

    dataset_train = ds_t.map(preprocess_function, batched=True)
    dataset_validation = ds_v.map(preprocess_function, batched=True)

    # Compute accuracy and F1-score, and track misclassified samples
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        acc = accuracy_score(labels, predictions)
        f1 = f1_score(labels, predictions, average='macro')
        

        # Identify misclassified samples
        misclassified_samples = pd.DataFrame({
            "News_Article": ds_v["news_title"],
            "True_Label": labels,
            "Predicted_Label": predictions
        })[labels != predictions]

        # Tokenize misclassified samples for analysis
        tokenized_output = tokenizer(misclassified_samples["News_Article"].tolist(), 
                                     padding=True, truncation=True, 
                                     max_length=max_sequence_length, return_tensors="pt")

        # Convert token IDs to actual tokens
        misclassified_samples["Tokenized_Article"] = [
            " ".join(tokenizer.convert_ids_to_tokens(token_ids)) 
            for token_ids in tokenized_output["input_ids"].tolist()
        ]

        # Ensure the directory exists
        output_dir = f"misclassified_{model_name}"
        os.makedirs(output_dir, exist_ok=True)

        # Save misclassified samples
        misclassified_samples.to_csv(f"{output_dir}/misclassified.csv", index=False)

        # Log results
        with open(log_file, 'a') as f:
            f.write(f"{model_name},{acc:.4f},{f1:.4f}\n")

        return {'accuracy': acc, 'f1_score': f1}

    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"{model_name}_output/", overwrite_output_dir=True, num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size, save_steps=save_steps,
        save_total_limit=1,fp16=True,learning_rate=1e-4,logging_steps=50,evaluation_strategy="steps",
        eval_steps=50,seed=42  # Set the seed for reproducibility
    )

    # Trainer
    trainer = Trainer(model=model,args=training_args,
        train_dataset=dataset_train,eval_dataset=dataset_validation,compute_metrics=compute_metrics
    )

    # Train model
    trainer.train()
    print(f"Training completed for model: {model_name}\n")

In [23]:
train_model('bert_wordpiece/')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert_wordpiece/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training Model: bert_wordpiece/


Map:   0%|          | 0/51227 [00:00<?, ? examples/s]

Map:   0%|          | 0/12807 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Accuracy,F1 Score
50,1.624200,1.309434,0.581088,0.557213
100,1.191100,1.028756,0.662841,0.659075
150,0.979900,0.896858,0.692824,0.688409
200,0.862700,0.825311,0.712267,0.714588
250,0.802200,0.779432,0.727180,0.723380
300,0.748200,0.744052,0.741626,0.741189
350,0.706400,0.721806,0.747248,0.747089
400,0.694200,0.717438,0.746623,0.745276
450,0.661100,0.707536,0.750839,0.749197
500,0.644400,0.698515,0.753963,0.753758


Training completed for model: bert_wordpiece/



In [24]:
train_model('bert_wordpiece_50000/')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert_wordpiece_50000/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training Model: bert_wordpiece_50000/


Map:   0%|          | 0/51227 [00:00<?, ? examples/s]

Map:   0%|          | 0/12807 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Accuracy,F1 Score
50,1.640400,1.295679,0.603576,0.579075
100,1.176600,1.006634,0.674084,0.667760
150,0.954200,0.872326,0.699695,0.691966
200,0.834500,0.797698,0.725228,0.725878
250,0.775900,0.751382,0.733115,0.731480
300,0.711000,0.726474,0.739127,0.736097
350,0.674000,0.706786,0.746857,0.744456
400,0.660200,0.693790,0.749043,0.749492
450,0.615400,0.687930,0.756071,0.756512
500,0.607300,0.678448,0.760912,0.762615


Training completed for model: bert_wordpiece_50000/



In [25]:
train_model('bert_senpiece/')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert_senpiece/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training Model: bert_senpiece/


Map:   0%|          | 0/51227 [00:00<?, ? examples/s]

Map:   0%|          | 0/12807 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Accuracy,F1 Score
50,1.605600,1.297544,0.583119,0.558345
100,1.191000,1.051797,0.650191,0.644658
150,0.997200,0.932130,0.677754,0.670022
200,0.889800,0.854722,0.700242,0.700456
250,0.830500,0.813599,0.714843,0.710914
300,0.774300,0.787475,0.722261,0.717576
350,0.742300,0.764317,0.731709,0.728630
400,0.728500,0.753684,0.734989,0.734785
450,0.686800,0.747596,0.737800,0.736614
500,0.680200,0.736317,0.742328,0.742902


Training completed for model: bert_senpiece/



In [26]:
train_model('bert_senpiece_50000/')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert_senpiece_50000/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training Model: bert_senpiece_50000/


Map:   0%|          | 0/51227 [00:00<?, ? examples/s]

Map:   0%|          | 0/12807 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Accuracy,F1 Score
50,1.599800,1.304687,0.582572,0.559045
100,1.195400,1.049297,0.660420,0.656054
150,0.995500,0.926419,0.687437,0.681754
200,0.872400,0.843755,0.713048,0.713644
250,0.807500,0.794767,0.725463,0.724121
300,0.742300,0.774317,0.731631,0.728421
350,0.711800,0.755458,0.736550,0.734494
400,0.696100,0.739569,0.742094,0.742072
450,0.652000,0.736639,0.745764,0.745771
500,0.641600,0.724829,0.748575,0.749670


Training completed for model: bert_senpiece_50000/



In [29]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
import os


# Model training parameters
max_sequence_length = 64
epochs = 5
batch_size = 64
save_steps = 1000

    
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# Training function
def train_model(model_name):
    print(f"\nTraining Model: {model_name}")
    
    log_file = f"{model_name}_results.csv"
    
    # Initialize logging
    with open(log_file, 'w') as f:
        f.write("Model,Accuracy,F1\n")
    
     # Set seed for reproducibility
    set_seed(42)

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=classes_num).to('cuda')
    
    # Preprocessing function
    def preprocess_function(examples):
        return tokenizer(examples['news_title'], truncation=True, padding="max_length", max_length=max_sequence_length)

    dataset_train = ds_t.map(preprocess_function, batched=True)
    dataset_validation = ds_v.map(preprocess_function, batched=True)

    # Compute accuracy and F1-score, and track misclassified samples
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        acc = accuracy_score(labels, predictions)
        f1 = f1_score(labels, predictions, average='macro')

    # Identify misclassified samples
        misclassified_samples = pd.DataFrame({
        "News_Article": ds_v["news_title"],
        "True_Label": labels,
        "Predicted_Label": predictions
         })[labels != predictions]

    # Tokenize misclassified samples for analysis
        tokenized_output = tokenizer(misclassified_samples["News_Article"].tolist(), 
                                 padding=True, truncation=True, 
                                 max_length=max_sequence_length, return_tensors="pt")

    # Convert token IDs to actual tokens
       # Convert token IDs to actual readable Tamil text
        misclassified_samples["Decoded_Article"] = [
                              tokenizer.decode(token_ids, skip_special_tokens=True)  
                               for token_ids in tokenized_output["input_ids"].tolist()
                                 ]


    # Ensure the directory exists
        output_dir = f"misclassified_{model_name}"
        os.makedirs(output_dir, exist_ok=True)

    # Save misclassified samples
        misclassified_samples.to_csv(f"{output_dir}/misclassified.csv", index=False)

    # Log results
        with open(log_file, 'a') as f:
            f.write(f"{model_name},{acc},{f1}\n")

        return {'accuracy': acc, 'f1_score': f1}



    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"{model_name}_output/",
        overwrite_output_dir=True,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        save_steps=save_steps,
        save_total_limit=1,
        fp16=True,
        learning_rate=1e-4,
        logging_steps=50,
        evaluation_strategy="steps",
        eval_steps=50,
        seed=42  # Set the seed for reproducibility
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset_train,
        eval_dataset=dataset_validation,
        compute_metrics=compute_metrics
    )

    # Train model
    trainer.train()


In [30]:
train_model('bert_bbpe/')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert_bbpe/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training Model: bert_bbpe/


Map:   0%|          | 0/51227 [00:00<?, ? examples/s]

Map:   0%|          | 0/12807 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Accuracy,F1 Score
50,1.845800,1.753245,0.306863,0.245331
100,1.695400,1.608690,0.374561,0.311890
150,1.563500,1.456678,0.459749,0.407426
200,1.445300,1.360710,0.494573,0.453018
250,1.372500,1.294447,0.520106,0.478786
300,1.316600,1.241545,0.541579,0.513135
350,1.259400,1.211389,0.556102,0.532875
400,1.237900,1.175694,0.572890,0.555177
450,1.194600,1.158031,0.583119,0.564795
500,1.171500,1.140826,0.584993,0.572728


In [31]:
train_model('bert_bbpe_50000/')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert_bbpe_50000/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training Model: bert_bbpe_50000/


Map:   0%|          | 0/51227 [00:00<?, ? examples/s]

Map:   0%|          | 0/12807 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss,Validation Loss,Accuracy,F1 Score
50,1.845500,1.754625,0.310299,0.261338
100,1.704300,1.633025,0.364879,0.309263
150,1.601900,1.523539,0.414539,0.368112
200,1.515300,1.466000,0.440618,0.400799
250,1.450400,1.405498,0.462403,0.419665
300,1.403400,1.337447,0.499102,0.463792
350,1.336200,1.293764,0.513547,0.482437
400,1.310200,1.265249,0.519716,0.485812
450,1.269200,1.235721,0.533693,0.504776
500,1.244200,1.216201,0.543609,0.519733


In [19]:
from transformers import BertTokenizer

# Load the pre-trained WordPiece tokenizer (BERT tokenizer in this case)
tokenizer = tokenizer = AutoTokenizer.from_pretrained('bert_bbpe/')

# Your Tamil text
text = 'பிரதமர் விருந்து:  தனுஷ் மகிழ்ச்சி'

# Tokenize the text
tokens = tokenizer.tokenize(text)

# Print the tokenized output and the length of the sequence
print(f"Tokenized Text: {tokens}")
print(f"Number of Tokens: {len(tokens)}")

# Check if it's within the limit (64 tokens)
if len(tokens) > 64:
    print("The sequence is too long and needs to be truncated or split.")
else:
    print("The sequence length is within the limit.")


Tokenized Text: ['à®ª', 'à®¿', 'à®°à®¤à®®à®°', 'à¯į', 'Ġà®µ', 'à®¿', 'à®°', 'à¯ģ', 'à®¨', 'à¯į', 'à®¤', 'à¯ģ:', 'Ġ', 'Ġà®¤à®©', 'à¯ģ', 'à®·', 'à¯į', 'Ġà®®à®ķ', 'à®¿', 'à®´', 'à¯į', 'à®ļ', 'à¯į', 'à®ļ', 'à®¿']
Number of Tokens: 25
The sequence length is within the limit.
